In [16]:
import s3fs, fsspec, datasets, pyarrow, boto3, botocore, sys
print("s3fs:", s3fs.__version__, s3fs.__file__)
print("fsspec:", fsspec.__version__, fsspec.__file__)
print("datasets:", datasets.__version__)
print("pyarrow:", pyarrow.__version__)
print("boto3/botocore:", boto3.__version__, botocore.__version__)

s3fs: 2025.3.0 C:\Users\karol\miniconda3\envs\ai_lawyer_project_tttt\lib\site-packages\s3fs\__init__.py
fsspec: 2025.3.0 C:\Users\karol\miniconda3\envs\ai_lawyer_project_tttt\lib\site-packages\fsspec\__init__.py


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│   1 import s3fs, fsspec, datasets, pyarrow, boto3, botocore, sys                                 │
│   2 print("s3fs:", s3fs.__version__, s3fs.__file__)                                              │
│   3 print("fsspec:", fsspec.__version__, fsspec.__file__)                                        │
│ ❱ 4 print("datasets:", datasets.__version__)                                                     │
│   5 print("pyarrow:", pyarrow.__version__)                                                       │
│   6 print("boto3/botocore:", boto3.__version__, botocore.__version__)                            │
│   7                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
AttributeError: module 'datasets' has no attribute '__version__'

In [1]:
!pip install -U "s3fs>=2024.6.1" "fsspec>=2024.6.1" "datasets>=2.20" "pyarrow>=14" "boto3>=1.34" "botocore>=1.34"


  Using cached s3fs-2025.7.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)
  Using cached boto3-1.40.6-py3-none-any.whl.metadata (6.7 kB)
  Using cached botocore-1.40.6-py3-none-any.whl.metadata (5.7 kB)
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-3.5.1-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-3.5.0-py3-none-any.whl.metadata (19 kB)
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached datasets-3.4.1-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-3.4.0-py3-none-any.whl.metadata (19 kB)
INFO: This is taking longer than usual. You might need to prov

In [2]:
!pip install -U "sagemaker>=2.190.0" "datasets>=2.20"

  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
Using cached datasets-4.0.0-py3-none-any.whl (494 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 2.18.0
    Uninstalling datasets-2.18.0:
      Successfully uninstalled datasets-2.18.0


In [1]:
from preprocessing.utils.defaults import AWS_REGION
import sagemaker
import boto3

sess = sagemaker.Session()

sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client('iam')
    role = iam.get_role(RoleName='sagemaker_execution_role')['Role']['Arn']

sess = sagemaker.Session(boto_session=boto3.Session(region_name=AWS_REGION), default_bucket=sagemaker_session_bucket)

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\karol\AppData\Local\sagemaker\sagemaker\config.yaml


Couldn't call 'get_role' to get Role ARN from role name superUser to get Role path.


sagemaker role arn: arn:aws:iam::767427092061:role/sagemaker_execution_role
sagemaker bucket: sagemaker-eu-central-1-767427092061
sagemaker session region: eu-west-1


In [2]:
from datasets import load_dataset


s3_dir =  's3://datalake-bucket-123/stages/$4b7047db-ab3c-4fca-811c-78069268dcae/ExplodeQuestionToQuestionChunkPair/results.parquet.gzip'

ds = load_dataset("parquet", data_files={"data": f"{s3_dir}/**/*.parquet"}, split="data")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [3]:
from datasets import Dataset, DatasetDict


def build_datasets(type_of_chunking: str, test_size=0.15):
    if type_of_chunking == "article_based_chunking":
        # ------------------------------------------------------------------
        # 1️⃣  Build a unique-query list and split that list first
        # ------------------------------------------------------------------
        # create a mini-dataset with one column (id_query) and one row per query
        query_id_ds = Dataset.from_list(
            [{"id_query": qid} for qid in set(ds["id_query"])]
        )
        q_split = query_id_ds.train_test_split(test_size=test_size)  # keep it for testing
        train_qids = set(q_split["train"]["id_query"])
        test_qids = set(q_split["test"]["id_query"])

        train_ds = ds.filter(lambda ex: ex["id_query"] in train_qids and ex["type"] == "Article_Span")
        test_ds = ds.filter(lambda ex: ex["id_query"] in test_qids and ex["type"] != "Article")
        return train_ds, test_ds
    elif type_of_chunking == "legal_unit_chunking":
        # ------------------------------------------------------------------
        # 1️⃣  Keep only rows that are exact legal units which are need to answer question
        # ------------------------------------------------------------------
        full_ds_with_legal_unit_chunking = ds.filter(lambda ex: ex["contains_citation"] == 0)

        # ------------------------------------------------------------------
        # 1️⃣  Build a unique-query list and split that list first
        # ------------------------------------------------------------------
        # create a mini-dataset with one column (id_query) and one row per query
        query_id_ds = Dataset.from_list(
            [{"id_query": qid} for qid in set(full_ds_with_legal_unit_chunking["id_query"])]
        )
        q_split = query_id_ds.train_test_split(test_size=test_size)  # keep it for testing
        train_qids = set(q_split["train"]["id_query"])
        test_qids = set(q_split["test"]["id_query"])

        train_ds = full_ds_with_legal_unit_chunking.filter(lambda ex: ex["id_query"] in train_qids and ex["type"] != "Article_Span")
        test_ds = full_ds_with_legal_unit_chunking.filter(lambda ex: ex["id_query"] in test_qids and ex["type"] != "Article_Span")

        return train_ds, test_ds


type_w = "legal_unit_chunking"



In [10]:
train_ds, test_ds = build_datasets("legal_unit_chunking")

Filter:   0%|          | 0/80935 [00:00<?, ? examples/s]

Filter:   0%|          | 0/80935 [00:00<?, ? examples/s]

In [4]:

train_ds_article_span, test_ds_article_span = build_datasets("article_based_chunking")

Filter:   0%|          | 0/148332 [00:00<?, ? examples/s]

Filter:   0%|          | 0/148332 [00:00<?, ? examples/s]

In [11]:
len(train_ds_article_span)

57243

In [12]:
len(test_ds_article_span)

18037

In [5]:
input_path = f's3://{sess.default_bucket()}/datasets/embedding_article_based_chunking'

In [6]:
train_ds_article_span.to_json(f"{input_path}/train/dataset.json", orient="records")
train_dataset_article_span_s3_path = f"{input_path}/train/dataset.json"
test_ds_article_span.to_json(f"{input_path}/test/dataset.json", orient="records")
test_dataset_article_span_s3_path = f"{input_path}/test/dataset.json"

Creating json from Arrow format:   0%|          | 0/58 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/19 [00:00<?, ?ba/s]

In [19]:
print(f"https://s3.console.aws.amazon.com/s3/buckets/{sess.default_bucket()}/?region={sess.boto_region_name}&prefix={input_path.split('/', 3)[-1]}/")


https://s3.console.aws.amazon.com/s3/buckets/sagemaker-eu-central-1-767427092061/?region=eu-west-1&prefix=datasets/embedding_article_based_chunking/


In [5]:
from sentence_transformers.evaluation import (
    InformationRetrievalEvaluator,
    SequentialEvaluator,
)
from sentence_transformers.util import cos_sim
from datasets import load_dataset, concatenate_datasets
from collections import defaultdict


def build_gold_evaluator(
        train_ds,
        test_ds,
        mode: str = "gold_unit_article_chunking",  # or "gold_article_only"
):
    """
    Parameters
    ----------
    train_ds, test_ds : datasets.Dataset
        Splits returned by build_datasets().
    mode : str
        • "gold_unit_article_chunking"  → Gold-Unit recall counts *all* chunks
          whose `contains_citation == 0`.  Corpus = every unique chunk
          (articles + sections + points).

        • "gold_article_only"           → Recall counts **only** rows that are
          *both* `type=="Article"` and `contains_citation==0`.  Corpus = article
          chunks only.  Queries that cite only §/pkt are ignored.
    emb_dim : int
        The embedding dimension printed in the metric names.
    """
    # ------------------------------------------------------------------
    # 1️⃣  Build CORPUS  (depends on mode)
    # ------------------------------------------------------------------
    corpus = {}
    seen = set()
    for row in concatenate_datasets([train_ds, test_ds]):
        if mode == "gold_article_only" and row["type"] != "Article_Span":
            continue  # skip §/pkt chunks in corpus
        cid = row["id_positive"]
        if cid not in seen:
            corpus[cid] = row["positive"]
            seen.add(cid)

    # ------------------------------------------------------------------
    # 2️⃣  Build QUERIES  (first anchor per id_query from test_ds)
    # ------------------------------------------------------------------
    queries = {}
    for row in test_ds:
        qid, anchor = row["id_query"], row["anchor"]
        queries.setdefault(qid, anchor)  # keep first (identical) anchor

    # ------------------------------------------------------------------
    # 3️⃣  Build RELEVANT_DOCS  according to mode
    # ------------------------------------------------------------------
    rel = defaultdict(list)

    for row in test_ds:
        if mode == "gold_article_only" and row["type"] != "Article_Span":
            # skip §/pkt rows in strict article-only metric
            continue

        rel[row["id_query"]].append(row["id_positive"])

    # remove queries that ended up empty (only happens in article-only mode)
    relevant_docs = {
        q: list(set(cids))
        for q, cids in rel.items() if cids
    }

    # ------------------------------------------------------------------
    # 4️⃣  Build evaluator
    # ------------------------------------------------------------------
    name = "gold_article" if mode == "gold_article_only" else "gold_unit"
    return InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=f"{name}_dim_{1024}",
        score_functions={"cosine": cos_sim},
    )



In [6]:
def build_relevant_chunk_evaluator(train_ds,
                                   test_ds,
                                   chunking_mode: str = "article_based_chunking" # or "legal_unit_chunking"
                                   ):


    # ------------------------------------------------------------------
    # 1️⃣  Build CORPUS  (depends on mode)
    # ------------------------------------------------------------------
    corpus = {}
    seen = set()
    for row in concatenate_datasets([train_ds, test_ds]):

        if chunking_mode == "legal_unit_chunking" and row["contains_citation"] == 1:
            continue  # skip article based chunks for it
        if chunking_mode == "article_based_chunking" and row["type"] != "Article_Span":
            continue  # skip §/pkt chunks in corpus
        cid = row["id_positive"]
        if cid not in seen:
            corpus[cid] = row["positive"]
            seen.add(cid)

    # ------------------------------------------------------------------ #
    # 2️⃣  QUERIES  (first anchor per id_query from test split)
    # ------------------------------------------------------------------ #
    queries = {}
    for row in test_ds:
        queries.setdefault(row["id_query"], row["anchor"])

    # ------------------------------------------------------------------ #
    # 3️⃣  RELEVANT_DOCS according to the mode
    # ------------------------------------------------------------------ #
    rel = defaultdict(list)

    for row in test_ds:
        if chunking_mode == "article_based_chunking":
            # parent articles that contain the cited unit
            if row["type"] == "Article_Span":
                rel[row["id_query"]].append(row["id_positive"])
        elif chunking_mode == "legal_unit_chunking":
            # exact units (article OR section / point)
            if row["contains_citation"] == 0:
                rel[row["id_query"]].append(row["id_positive"])

    relevant_docs = {q: list(set(cids)) for q, cids in rel.items() if cids}

    # ------------------------------------------------------------------ #
    # 4.  Instantiate the evaluator
    # ------------------------------------------------------------------ #
    name = "relevant_chunk" if chunking_mode == "article_based_chunking" else "gold_unit"
    return InformationRetrievalEvaluator(
        queries        = queries,
        corpus         = corpus,
        relevant_docs  = relevant_docs,
        name           = f"{name}_dim_{1024}",
        score_functions={"cosine": cos_sim},
    )



In [11]:

contain_eval = build_relevant_chunk_evaluator(train_ds_article_span, test_ds_article_span,
                                              chunking_mode="article_based_chunking")




gold_unit_eval = build_relevant_chunk_evaluator(train_ds, test_ds,
                                                chunking_mode="legal_unit_chunking")

In [12]:
eval_gold_article = build_gold_evaluator(
    train_ds_article_span, test_ds_article_span, mode="gold_article_only"
)

eval_gold_unit = build_gold_evaluator(
    train_ds_article_span, test_ds_article_span, mode="gold_unit_article_chunking"
)

In [12]:
train_ds_article_span[0:5]

{'id_query': ['620773400', '620773400', '620779160', '620788980', '620792340'],
 'anchor': ['[query]: Jak należy zaksięgować w księgach rachunkowych błąd inwentaryzacji ujawniony obecnie w spółce jawnej, polegający na zawyżeniu pozycji „roboty w toku” na koniec 2014 r. w wyniku podwójnego zaksięgowania faktur (kwota >30 000 zł)? Jakie operacje księgowe i na jakich kontach należy wykonać oraz czy korektę należy zakwalifikować jako błąd poprzednich lat?',
  '[query]: Jak należy zaksięgować w księgach rachunkowych błąd inwentaryzacji ujawniony obecnie w spółce jawnej, polegający na zawyżeniu pozycji „roboty w toku” na koniec 2014 r. w wyniku podwójnego zaksięgowania faktur (kwota >30 000 zł)? Jakie operacje księgowe i na jakich kontach należy wykonać oraz czy korektę należy zakwalifikować jako błąd poprzednich lat?',
  '[query]: Czy od dopłat na kapitał rezerwowy wniesionych przez wspólników spółki z ograniczoną odpowiedzialnością, które mają zostać zwrócone do określonego terminu (do 30 

In [13]:
r = 4

In [4]:
print(test_ds[0:5])

{'id_query': ['620758395', '620758395', '620758395', '620758395', '620773400'], 'anchor': ['[query]: Czy zgodnie z ustawą o rachunkowości skutki kontroli polegające na zwiększeniu podatku dochodowego, podatku VAT oraz wartości środków trwałych, przy braku konieczności korekty deklaracji VAT i rozliczenia rocznego, należy rozliczyć w wyniku finansowym roku bieżącego? Jeśli tak, w jaki sposób ująć księgowo: (i) korektę podatku VAT, (ii) korektę inwestycji w obcych środkach trwałych (zwiększenie wartości środków trwałych) oraz (iii) korektę podatku dochodowego?', '[query]: Czy zgodnie z ustawą o rachunkowości skutki kontroli polegające na zwiększeniu podatku dochodowego, podatku VAT oraz wartości środków trwałych, przy braku konieczności korekty deklaracji VAT i rozliczenia rocznego, należy rozliczyć w wyniku finansowym roku bieżącego? Jeśli tak, w jaki sposób ująć księgowo: (i) korektę podatku VAT, (ii) korektę inwestycji w obcych środkach trwałych (zwiększenie wartości środków trwałych)

In [7]:
print(test_ds[0:50])

{'id_query': ['620758395', '620758395', '620758395', '620758395', '620773400', '620773400', '620773400', '620781510', '620781510', '620781510', '620781510', '620781510', '620787810', '620787810', '620787810', '620787810', '620787810', '620787810', '620788980', '620788980', '620788980', '620789170', '620789170', '620789170', '620789170', '620789170', '620789170', '620789170', '620789170', '620789170', '620789170', '620789170', '620789170', '620789170', '620789170', '620789170', '620791975', '620791975', '620791975', '620791975', '620793125', '620793125', '620793125', '620793125', '620793125', '620793125', '620793125', '620793125', '620793125', '620793125'], 'anchor': ['[query]: Czy zgodnie z ustawą o rachunkowości skutki kontroli polegające na zwiększeniu podatku dochodowego, podatku VAT oraz wartości środków trwałych, przy braku konieczności korekty deklaracji VAT i rozliczenia rocznego, należy rozliczyć w wyniku finansowym roku bieżącego? Jeśli tak, w jaki sposób ująć księgowo: (i) ko

In [9]:
import time
from sagemaker.huggingface import HuggingFace

training_arguments = {
    "model_id": "sdadas/mmlw-retrieval-roberta-large-v2",  # model id from the hub
    "chunking_type":"article_based_chunking",
    "train_dataset_path": "/opt/ml/input/data/train/",  # path inside the container where the training data is stored
    "test_dataset_path": "/opt/ml/input/data/test/",  # path inside the container where the test data is stored
    "num_train_epochs": 5,  # number of training epochs
    "learning_rate": 2e-5,  # learning rate
    'per_device_train_batch_size': 6,  # batch size per device during training
    'per_device_eval_batch_size': 4,
    'gradient_accumulation_steps': 8
}

job_name = f'roberta-large-{time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())}-article-span-retrival'

# create the Estimator
huggingface_estimator = HuggingFace(
    entry_point='run_mnr.py',  # train script
    source_dir='scripts_pirb',  # directory which includes all the files needed for training
    instance_type='ml.g5.xlarge',  # instances type used for the training job
    instance_count=1,  # the number of instances used for training
    max_run=2 * 24 * 60 * 60,  # maximum runtime in seconds (days * hours * minutes * seconds)
    base_job_name=job_name,  # the name of the training job
    role=role,  # Iam role used in training job to access AWS ressources, e.g. S3
    transformers_version='4.36.0',  # the transformers version used in the training job
    pytorch_version='2.1.0',  # the pytorch_version version used in the training job
    py_version='py310',  # the python version used in the training job
    hyperparameters=training_arguments,
    disable_output_compression=True,  # not compress output to save training time and cost
    environment={
        "HUGGINGFACE_HUB_CACHE": "/tmp/.cache",  # set env variable to cache models in /tmp
    },
)

In [10]:
data = {
    'train': train_dataset_article_span_s3_path,
    'test': test_dataset_article_span_s3_path,
}

# starting the train job with our uploaded datasets as input
huggingface_estimator.fit(data, wait=True)

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: roberta-large-2025-08-11-01-06-53-artic-2025-08-10-23-06-57-749


2025-08-10 23:07:04 Starting - Starting the training job
2025-08-10 23:07:04 Pending - Training job waiting for capacity...
2025-08-10 23:07:18 Pending - Preparing the instances for training...
2025-08-10 23:07:49 Downloading - Downloading input data...
2025-08-10 23:08:10 Downloading - Downloading the training image........................
2025-08-10 23:12:28 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will b

In [13]:
model_artifact_s3 = huggingface_estimator.model_data
print(model_artifact_s3)  # s3://.../<job-name>/output/model.tar.gz

{'S3DataSource': {'S3Uri': 's3://sagemaker-eu-central-1-767427092061/roberta-large-2025-08-11-01-06-53-artic-2025-08-10-23-06-57-749/output/model/', 'S3DataType': 'S3Prefix', 'CompressionType': 'None'}}


In [18]:
train_ds_exact_legal_unit, test_ds_exact_legal_unit = build_datasets("legal_unit_chunking")


Filter:   0%|          | 0/80935 [00:00<?, ? examples/s]

Filter:   0%|          | 0/80935 [00:00<?, ? examples/s]

In [16]:
len(train_ds_exact_legal_unit)

67784

In [17]:
len(test_ds_exact_legal_unit)

13151

In [20]:
input_path_exact_legal_unit = f's3://{sess.default_bucket()}/datasets/embedding_legal_unit_chunking'

In [21]:
train_ds_exact_legal_unit.to_json(f"{input_path_exact_legal_unit}/train/dataset.json", orient="records")
train_ds_exact_legal_unit_s3_path = f"{input_path_exact_legal_unit}/train/dataset.json"


test_ds_exact_legal_unit.to_json(f"{input_path_exact_legal_unit}/test/dataset.json", orient="records")
test_ds_exact_legal_unit_s3_path = f"{input_path_exact_legal_unit}/test/dataset.json"

Creating json from Arrow format:   0%|          | 0/70 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/12 [00:00<?, ?ba/s]

In [22]:
import time
from sagemaker.huggingface import HuggingFace

training_arguments_for_exact_legal_unit = {
    "model_id": "sdadas/mmlw-retrieval-roberta-large-v2",  # model id from the hub
    "chunking_type": "legal_unit_chunking",
    "train_dataset_path": "/opt/ml/input/data/train/",  # path inside the container where the training data is stored
    "test_dataset_path": "/opt/ml/input/data/test/",  # path inside the container where the test data is stored
    "num_train_epochs": 5,  # number of training epochs
    "learning_rate": 2e-5,  # learning rate
    'per_device_train_batch_size': 6,  # batch size per device during training
    'per_device_eval_batch_size': 4,
    'gradient_accumulation_steps': 8
}

job_name = f'roberta-{time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())}-exact-legal-unit'

# create the Estimator
huggingface_estimator_for_exact_legal_unit = HuggingFace(
    entry_point='run_mnr.py',  # train script
    source_dir='scripts_pirb',  # directory which includes all the files needed for training
    instance_type='ml.g5.xlarge',  # instances type used for the training job
    instance_count=1,  # the number of instances used for training
    max_run=2 * 24 * 60 * 60,  # maximum runtime in seconds (days * hours * minutes * seconds)
    base_job_name=job_name,  # the name of the training job
    role=role,  # Iam role used in training job to access AWS ressources, e.g. S3
    transformers_version='4.36.0',  # the transformers version used in the training job
    pytorch_version='2.1.0',  # the pytorch_version version used in the training job
    py_version='py310',  # the python version used in the training job
    hyperparameters=training_arguments_for_exact_legal_unit,
    disable_output_compression=True,  # not compress output to save training time and cost
    environment={
        "HUGGINGFACE_HUB_CACHE": "/tmp/.cache",  # set env variable to cache models in /tmp
    },
)

In [23]:


data_for_exact = {
    'train': train_ds_exact_legal_unit_s3_path,
    'test': test_ds_exact_legal_unit_s3_path,
}

# starting the train job with our uploaded datasets as input
huggingface_estimator_for_exact_legal_unit.fit(data_for_exact, wait=True)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: roberta-2025-08-11-11-25-36-exact-legal-2025-08-11-09-25-44-663


2025-08-11 09:25:46 Starting - Starting the training job
2025-08-11 09:25:46 Pending - Training job waiting for capacity...
2025-08-11 09:26:08 Pending - Preparing the instances for training...
2025-08-11 09:26:35 Downloading - Downloading input data...
2025-08-11 09:26:55 Downloading - Downloading the training image........................
2025-08-11 09:31:18 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will b

In [ ]:
import time
from sagemaker.huggingface import HuggingFace

training_arguments_for_exact_legal_unit = {
    "model_id": "sdadas/mmlw-retrieval-roberta-large-v2",  # model id from the hub
    "chunking_type": "legal_unit_chunking",
    "train_dataset_path": "/opt/ml/input/data/train/",  # path inside the container where the training data is stored
    "test_dataset_path": "/opt/ml/input/data/test/",  # path inside the container where the test data is stored
    "num_train_epochs": 5,  # number of training epochs
    "learning_rate": 2e-5,  # learning rate
    'per_device_train_batch_size': 6,  # batch size per device during training
    'per_device_eval_batch_size': 4,
    'gradient_accumulation_steps': 8
}

job_name = f'roberta-{time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())}-exact-legal-unit'

# create the Estimator
huggingface_estimator_for_exact_legal_unit_with_added = HuggingFace(
    entry_point='run_mnr.py',  # train script
    source_dir='scripts_pirb',  # directory which includes all the files needed for training
    instance_type='ml.g5.xlarge',  # instances type used for the training job
    instance_count=1,  # the number of instances used for training
    max_run=2 * 24 * 60 * 60,  # maximum runtime in seconds (days * hours * minutes * seconds)
    base_job_name=job_name,  # the name of the training job
    role=role,  # Iam role used in training job to access AWS ressources, e.g. S3
    transformers_version='4.36.0',  # the transformers version used in the training job
    pytorch_version='2.1.0',  # the pytorch_version version used in the training job
    py_version='py310',  # the python version used in the training job
    hyperparameters=training_arguments_for_exact_legal_unit,
    disable_output_compression=True,  # not compress output to save training time and cost
    environment={
        "HUGGINGFACE_HUB_CACHE": "/tmp/.cache",  # set env variable to cache models in /tmp
    },
)